# Blind Testing Setup: Dec/Jan vs Feb

This notebook prepares a time-based blind-testing view for the transaction dataset.

Dataset period:
- Train/reference period: `2024-12-30` to `2025-01-31`
- Blind test period: `2025-02-01` to `2025-02-15`

Questions answered here:
- How many labeled points/accounts do we have overall?
- How many labeled accounts are active in Dec/Jan?
- How many labeled accounts are active in Feb?
- How much overlap exists between the two periods?
- What does the entity distribution look like in each period?

In [2]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

In [3]:
DATA_DIR = Path('.')
LABELS_PATH = DATA_DIR / '../labeled-data/normalization/labels_mapped_normalized.csv'
TX_PATH = DATA_DIR / 'transaction_edges_minimal.csv'

TRAIN_START = '2024-12-30'
TRAIN_END = '2025-01-31'
TEST_START = '2025-02-01'
TEST_END = '2025-02-15'

CHUNK_SIZE = 2_000_000

LABELS_PATH, TX_PATH

(PosixPath('../labeled-data/normalization/labels_mapped_normalized.csv'),
 PosixPath('transaction_edges_minimal.csv'))

## 1. Load Labeled Accounts

In [4]:
labels = pd.read_csv(LABELS_PATH, usecols=['account_id', 'name']).dropna().drop_duplicates()
labels['account_id'] = labels['account_id'].astype('int64')
labels = labels.rename(columns={'name': 'entity'})

label_ids = set(labels['account_id'])
label_map = dict(zip(labels['account_id'], labels['entity']))

print(f'Total labeled accounts: {len(labels):,}')
print(f'Unique entities: {labels["entity"].nunique():,}')
labels.head()

Total labeled accounts: 8,336
Unique entities: 212


,account_id,entity
0,563998,SCAM
1,403,AQUA
2,22177,Coinbase
3,28760,Coinbase
4,17577247,SCAM


In [5]:
labels['entity'].value_counts().head(15).rename_axis('entity').reset_index(name='n_accounts')

,entity,n_accounts
0,SCAM,7784
1,EXLM,185
2,Coinbase,38
3,SDF,20
4,Wirex,19
5,AQUA,11
6,UltraCapital,8
7,Binance,8
8,Imposter,8
9,Lobstr,6


## 2. Stream Transactions and Split by Period

In [6]:
period_accounts = {
    'dec_jan': set(),
    'feb': set(),
    'full': set(),
}

period_tx_rows = {
    'dec_jan': 0,
    'feb': 0,
    'full': 0,
}

date_counts = {}

chunks = pd.read_csv(
    TX_PATH,
    usecols=['sender_id', 'receiver_id', 'tx_date'],
    chunksize=CHUNK_SIZE,
)

for i, chunk in enumerate(chunks, start=1):
    chunk = chunk.dropna(subset=['sender_id', 'receiver_id', 'tx_date']).copy()
    chunk['sender_id'] = chunk['sender_id'].astype('int64')
    chunk['receiver_id'] = chunk['receiver_id'].astype('int64')

    vc = chunk['tx_date'].value_counts()
    for day, count in vc.items():
        date_counts[day] = date_counts.get(day, 0) + int(count)

    dec_jan_mask = (chunk['tx_date'] >= TRAIN_START) & (chunk['tx_date'] <= TRAIN_END)
    feb_mask = (chunk['tx_date'] >= TEST_START) & (chunk['tx_date'] <= TEST_END)
    full_mask = dec_jan_mask | feb_mask

    for period_name, mask in [('dec_jan', dec_jan_mask), ('feb', feb_mask), ('full', full_mask)]:
        sub = chunk.loc[mask, ['sender_id', 'receiver_id']]
        period_tx_rows[period_name] += len(sub)
        if len(sub):
            active_accounts = set(sub['sender_id']) | set(sub['receiver_id'])
            period_accounts[period_name].update(active_accounts & label_ids)

    if i % 5 == 0:
        print(f'Processed {i:,} chunks...')

print('Done.')

Processed 5 chunks...
Processed 10 chunks...
Processed 15 chunks...
Processed 20 chunks...
Processed 25 chunks...
Processed 30 chunks...
Processed 35 chunks...
Processed 40 chunks...
Processed 45 chunks...
Done.


## 3. Overall Time Coverage

In [7]:
date_counts_df = (
    pd.Series(date_counts)
    .rename_axis('tx_date')
    .reset_index(name='n_transactions')
    .sort_values('tx_date')
    .reset_index(drop=True)
)

print('Min date:', date_counts_df['tx_date'].min())
print('Max date:', date_counts_df['tx_date'].max())
print('Distinct dates:', len(date_counts_df))

date_counts_df

Min date: 2024-12-30
Max date: 2025-02-15
Distinct dates: 31


,tx_date,n_transactions
0,2024-12-30,199203
1,2024-12-31,3615428
2,2025-01-01,30257225
3,2025-01-02,12906300
4,2025-01-03,2051238
5,2025-01-04,1996773
6,2025-01-05,1981456
7,2025-01-06,2235163
8,2025-01-07,2951047
9,2025-01-08,2881015


## 4. Blind Testing Summary

In [8]:
dec_jan_accounts = period_accounts['dec_jan']
feb_accounts = period_accounts['feb']
full_accounts = period_accounts['full']

summary = pd.DataFrame([
    {
        'period': 'all_labeled_accounts',
        'n_transactions': pd.NA,
        'n_active_labeled_accounts': len(label_ids),
    },
    {
        'period': 'dec_jan',
        'n_transactions': period_tx_rows['dec_jan'],
        'n_active_labeled_accounts': len(dec_jan_accounts),
    },
    {
        'period': 'feb',
        'n_transactions': period_tx_rows['feb'],
        'n_active_labeled_accounts': len(feb_accounts),
    },
    {
        'period': 'full_window',
        'n_transactions': period_tx_rows['full'],
        'n_active_labeled_accounts': len(full_accounts),
    },
    {
        'period': 'overlap_dec_jan_and_feb',
        'n_transactions': pd.NA,
        'n_active_labeled_accounts': len(dec_jan_accounts & feb_accounts),
    },
    {
        'period': 'feb_only',
        'n_transactions': pd.NA,
        'n_active_labeled_accounts': len(feb_accounts - dec_jan_accounts),
    },
    {
        'period': 'dec_jan_only',
        'n_transactions': pd.NA,
        'n_active_labeled_accounts': len(dec_jan_accounts - feb_accounts),
    },
])

summary

,period,n_transactions,n_active_labeled_accounts
0,all_labeled_accounts,<NA>,8336
1,dec_jan,92734837,8335
2,feb,808399,410
3,full_window,93543236,8336
4,overlap_dec_jan_and_feb,<NA>,409
5,feb_only,<NA>,1
6,dec_jan_only,<NA>,7926


Interpretation:
- `feb` gives the number of labeled points available for blind testing.
- `overlap_dec_jan_and_feb` tells us how many labeled accounts appear in both periods.
- `feb_only` tells us how many labeled Feb accounts are unseen in Dec/Jan.

## 5. Entity Distribution by Period

In [9]:
def entity_counts_for_accounts(account_ids):
    return (
        pd.Series([label_map[a] for a in account_ids], name='entity')
        .value_counts()
        .rename_axis('entity')
        .reset_index(name='n_accounts')
    )

entity_dec_jan = entity_counts_for_accounts(dec_jan_accounts).rename(columns={'n_accounts': 'dec_jan_accounts'})
entity_feb = entity_counts_for_accounts(feb_accounts).rename(columns={'n_accounts': 'feb_accounts'})

entity_compare = (
    entity_dec_jan.merge(entity_feb, on='entity', how='outer')
    .fillna(0)
)
entity_compare['dec_jan_accounts'] = entity_compare['dec_jan_accounts'].astype(int)
entity_compare['feb_accounts'] = entity_compare['feb_accounts'].astype(int)
entity_compare['delta_feb_minus_dec_jan'] = entity_compare['feb_accounts'] - entity_compare['dec_jan_accounts']
entity_compare = entity_compare.sort_values(['feb_accounts', 'dec_jan_accounts'], ascending=False).reset_index(drop=True)

entity_compare.head(30)

,entity,dec_jan_accounts,feb_accounts,delta_feb_minus_dec_jan
0,SCAM,7783,296,-7487
1,Coinbase,38,32,-6
2,SDF,20,11,-9
3,Binance,8,6,-2
4,TMM bot,4,4,0
5,EXLM,185,3,-182
6,Wirex,19,3,-16
7,AQUA,11,3,-8
8,UltraCapital,8,3,-5
9,Lobstr,6,3,-3


In [10]:
entity_compare.query('feb_accounts > 0 and dec_jan_accounts == 0').reset_index(drop=True).head(30)

,entity,dec_jan_accounts,feb_accounts,delta_feb_minus_dec_jan


## 6. Recommended Blind Testing Logic

Suggested protocol:

1. Build the graph, embeddings, and communities using only Dec/Jan transactions.
2. Keep Feb labeled accounts hidden during model or parameter selection.
3. Evaluate on Feb by checking whether Feb labeled accounts fall into coherent communities or match the expected entity structure.
4. Report two views:
   - All Feb labeled accounts
   - Feb-only accounts that were not seen in Dec/Jan

The second view is the stronger blind test because it reduces leakage from repeated accounts.